**Step 1 — Install & Import Everything**

In [1]:
import torch
import torch.nn as nn
import re
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader

**Step 2 - Text Preprocessing**

In [2]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text.split()

def tokenize(text):
    return clean_text(text)

#Test
print(tokenize("Deepika loves working at Google!"))

['deepika', 'loves', 'working', 'at', 'google']


**Step 3- Sentiment Model**

In [16]:
#_______Dataset_________________________________
sentiment_texts = [
    "i love this", 
    "this is amazing", 
    "wonderful experience",
    "best thing ever", 
    "i really enjoyed this", 
    "absolutely fantastic",
    "i hate this", 
    "this is terrible", 
    "worst experience ever",
    "absolutely disgusting", 
    "never buying this again", 
    "very disappointing",
    "not good at all", 
    "really bad", 
    "i dislike this",
    "i love this product",
    "i am loving this",
    "this is great",
    "i enjoy this",
    "this is very good",
    "i am happy",
    "i hate this food",
    "this is bad",
    "very disappointing choice",
    "i really hate this",
    "i hate working",
    "i hate waiting",
    "this is very bad",
    "this is terrible",
    "i dislike this product",
    "i am unhappy",
    "i hate meetings",
    "i hate delays",
    "deepika hates delays",
    "people hate late meetings",
    "i hate late comers",
    "this meeting is bad"
]

sentiment_labels = [1,1,1,1,1,1, 0,0,0,0,0,0,0,0,0, 1,1,1,1,1,1,1,0,0,0, 0,0,0,0,0,0,0,0,0,0, 0,0,0]

#________Build Vocab_________________________________
all_words = []
for t in sentiment_texts:
    all_words.extend(tokenize(t))

word_counts = Counter(all_words)
sent_vocab = ["<PAD>","<UNK>"] + [w for w, _ in word_counts.most_common()]
sent_word2idx ={w: i for i, w in enumerate(sent_vocab)}

#________Encode________________________________________
def encode_sentiment(text, word2idx, max_len=10):
    tokens = tokenize(text.lower())
    encoded = [word2idx.get(w, 1) for w in tokens]
    encoded += [0] * (max_len - len(encoded))

    return encoded[:max_len]

#_________Model________________________________________
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(SentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self,x):
        embedded = self.embedding(x)
        _, (hidden,cell ) = self.lstm(embedded)
        hidden = self.dropout(hidden.squeeze(0))
        return self.sigmoid(self.fc(hidden)).squeeze(1)

#_________Train___________________________________________
sent_dataset = []
for text, label in zip(sentiment_texts, sentiment_labels):
    x = torch.tensor(encode_sentiment(text, sent_word2idx), dtype=torch.long)
    y = torch.tensor(label, dtype=torch.float)
    sent_dataset.append((x,y))

sent_loader = DataLoader(sent_dataset, batch_size=4, shuffle=True)

sent_model = SentimentLSTM(len(sent_vocab), 16, 32)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(sent_model.parameters(), lr=0.001)

print("Trining Sentiment Model...")
for epoch in range(100):
    sent_model.train()
    for x_batch, y_batch in sent_loader:
        optimizer.zero_grad()
        loss = criterion(sent_model(x_batch), y_batch)
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 20:
        print(f" Epoch {epoch+1}/100 done")

print("Sentiment Model Ready!")

Trining Sentiment Model...
Sentiment Model Ready!


**Step 4 — NER Model**

In [17]:
# ── Dataset ───────────────────────────────────
ner_data = [
    (["Deepika", "works", "at", "Google", "in", "Hyderabad"],
     ["B-PER", "O", "O", "B-ORG", "O", "B-LOC"]),
    (["Elon", "Musk", "founded", "Tesla", "in", "California"],
     ["B-PER", "I-PER", "O", "B-ORG", "O", "B-LOC"]),
    (["Microsoft", "is", "based", "in", "Seattle"],
     ["B-ORG", "O", "O", "O", "B-LOC"]),
    (["Sundar", "Pichai", "leads", "Google", "in", "New", "York"],
     ["B-PER", "I-PER", "O", "B-ORG", "O", "B-LOC", "I-LOC"]),
    (["Priya", "studied", "at", "IIT", "in", "Mumbai"],
     ["B-PER", "O", "O", "B-ORG", "O", "B-LOC"]),
    (["Subhashree", "Peddi", "eating", "dinner", "in", "South", "Korea"],
     ["B-PER", "I-PER", "O", "O", "O", "B-LOC", "I-LOC"]),
    (["Deepika", "hate", "late", "comers", "in", "meetings"],
     ["B-PER",  "O",    "O",    "O",      "O",  "O"])
]

# ── Vocabulary ────────────────────────────────
ner_words = []
for s, _ in ner_data:
    ner_words.extend(s)

ner_word2idx = {"<PAD>":0,"<UNK>":1}
for w in set(ner_words):
    ner_word2idx[w] = len(ner_word2idx)

tag2idx = {
    "<PAD>":0,
    "O"    :1,
    "B-PER":2,
    "I-PER":3,
    "B-ORG":4,
    "I-ORG":5,
    "B-LOC":6,
    "I-LOC":7
}
idx2tag = {v: k for k, v in tag2idx.items()}

MAX_LEN = 10

def encoded_ner_sentence(sentence, word2idx, max_len=MAX_LEN):
    encoded = [word2idx.get(w, 1) for w in sentence]
    encoded += [0] * (max_len - len(encoded))
    return encoded[:max_len]

def encoded_ner_tag(tags, tag2idx, max_len=MAX_LEN):
    encoded = [tag2idx[t] for t in tags]
    encoded += [0] * (max_len - len(encoded))
    return encoded[:max_len]

#_______ Model_____________________________________
class NERModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_tags):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size,
                            hidden_size,
                            batch_first=True,
                            bidirectional=True
                           )
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size * 2, num_tags)

    def forward(self,x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        output = self.dropout(output)
        return self.fc(output)

#______Train____________________________________________________
ner_dataset = []
for sentence, tags in ner_data:
    x = torch.tensor(encoded_ner_sentence(sentence, ner_word2idx), dtype=torch.long)
    y = torch.tensor(encoded_ner_tag(tags, tag2idx), dtype=torch.long)
    ner_dataset.append((x,y))

    ner_dataloader = DataLoader(ner_dataset, batch_size=2, shuffle=True)
    ner_model = NERModel(len(ner_word2idx), 32, 16, len(tag2idx))
    ner_criterion = nn.CrossEntropyLoss(ignore_index=0)
    ner_optimizer = torch.optim.Adam(ner_model.parameters(), lr=0.001)

print("Training NER Model..")

for epoch in range(100):
    ner_model.train()
    total_loss = 0

    for x_batch, y_batch in ner_dataloader:
        ner_optimizer.zero_grad()

        logits = ner_model(x_batch)

        loss = ner_criterion(
            logits.reshape(-1, len(tag2idx)),
            y_batch.reshape(-1)
        )

        loss.backward()
        ner_optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/100 | Loss: {total_loss:.4f}")

print("NER Model Ready!")
        

Training NER Model..
Epoch 20/100 | Loss: 5.2754
Epoch 40/100 | Loss: 3.0035
Epoch 60/100 | Loss: 1.6473
Epoch 80/100 | Loss: 0.9176
Epoch 100/100 | Loss: 0.4859
NER Model Ready!


**Step 5 — Combined Pipeline Function**

In [19]:
def nlp_pipeline(text):
    print("=" * 50)
    print(f"📝 Input: {text}")
    print("=" * 50)

    # ── Sentiment ────────────────────────────────
    sent_model.eval()
    encoded = encode_sentiment(text, sent_word2idx)
    tensor  = torch.tensor([encoded], dtype=torch.long)

    with torch.no_grad():
        prob = sent_model(tensor).item()

    if prob > 0.65:
        sentiment = "Positive 😊"
    elif prob < 0.35:
        sentiment = "Negative 😞"
    else:
        sentiment = "Neutral 😐"
        
    confidence = round(prob * 100, 2)          # ✅ multiply by 100

    print(f"\n🎭 SENTIMENT ANALYSIS")
    print(f"   Sentiment  : {sentiment}")
    print(f"   Confidence : {confidence}%")    # ✅ shows 36.0% not 0.36%

    # ── NER ──────────────────────────────────────
    ner_model.eval()
    tokens  = text.split()
    encoded = encoded_ner_sentence(tokens, ner_word2idx)
    tensor  = torch.tensor([encoded], dtype=torch.long)

    with torch.no_grad():
        logits      = ner_model(tensor)
        predictions = logits.argmax(dim=2)[0]

    print(f"\n🏷️  NAMED ENTITY RECOGNITION")
    print(f"   {'Word':<15} {'Entity Tag'}")
    print(f"   {'-'*30}")

    entities = []
    for i, token in enumerate(tokens):
        tag    = idx2tag[predictions[i].item()]
        marker = "  ←" if tag != "O" else ""
        print(f"   {token:<15} {tag}{marker}")
        if tag != "O":
            entities.append((token, tag))

    print(f"\n✅ Entities Found: {entities}")
    print("=" * 50)


# ── Run all 4 tests ───────────────────────────────
nlp_pipeline("Deepika loves working at Google in Hyderabad")
nlp_pipeline("Elon Musk hates the product from Microsoft")
nlp_pipeline("Subhashree Peddi is enjoying dinner in South Korea")
nlp_pipeline("I love learning AI at Anthropic in Hyderabad")
nlp_pipeline("Deepika hate late comers in meetings")

📝 Input: Deepika loves working at Google in Hyderabad

🎭 SENTIMENT ANALYSIS
   Sentiment  : Positive 😊
   Confidence : 99.62%

🏷️  NAMED ENTITY RECOGNITION
   Word            Entity Tag
   ------------------------------
   Deepika         B-PER  ←
   loves           O
   working         B-ORG  ←
   at              O
   Google          B-ORG  ←
   in              O
   Hyderabad       B-LOC  ←

✅ Entities Found: [('Deepika', 'B-PER'), ('working', 'B-ORG'), ('Google', 'B-ORG'), ('Hyderabad', 'B-LOC')]
📝 Input: Elon Musk hates the product from Microsoft

🎭 SENTIMENT ANALYSIS
   Sentiment  : Positive 😊
   Confidence : 99.63%

🏷️  NAMED ENTITY RECOGNITION
   Word            Entity Tag
   ------------------------------
   Elon            B-PER  ←
   Musk            I-PER  ←
   hates           B-ORG  ←
   the             B-ORG  ←
   product         B-ORG  ←
   from            B-ORG  ←
   Microsoft       B-ORG  ←

✅ Entities Found: [('Elon', 'B-PER'), ('Musk', 'I-PER'), ('hates', 'B-ORG'), ('th